# Notebook 06 ? User Segmentation & Clustering

**Project:** Mirae Asset Digital Platform ? User Analytics  
**Phases covered:** Phase 12 (RFM Segmentation & K-Means Clustering)  
**Objective:** Move beyond averages. Group users into meaningful behavioural segments so that retention, marketing, and product decisions can be precisely targeted rather than broadcast.

| Section | Method | Output |
|---------|--------|--------|
| A | RFM Scoring | Rule-based segments: Champions, Loyal, At-Risk, Lost... |
| B | K-Means Clustering | Data-driven segments: VIP, Engaged, Casual, Dormant |
| C | Cluster Profiling | Demographics, behaviour, observable churn rate per segment |
| D | Business Actions | Retention playbook per segment |
| E | Feature Export | Cluster labels saved for NB07 predictive modelling |


## Table of Contents

**Part A — RFM Segmentation**  
A.1 [Build RFM Table from Transactions](#a1)  
A.2 [RFM Score Distribution](#a2)  
A.3 [Rule-Based Segment Assignment](#a3)  
A.4 [Segment Size & Revenue Profile](#a4)  
A.5 [RFM Heatmap (R vs F coloured by M)](#a5)  

**Part B — K-Means Clustering**  
B.1 [Feature Preparation & Scaling](#b1)  
B.2 [Optimal K — Elbow + Silhouette](#b2)  
B.3 [Fit K=4 & Assign Cluster Labels](#b3)  
B.4 [Cluster Scatter (PCA 2D)](#b4)  

**Part C — Cluster Deep-Dive Profiling**  
C.1 [Revenue & Frequency by Cluster](#c1)  
C.2 [Churn Rate & Engagement by Cluster](#c2)  
C.3 [Demographic Mix by Cluster](#c3)  
C.4 [Cluster Radar Chart (Multi-Metric)](#c4)  

**Part D — Business Strategy per Segment**  
D.1 [Retention Playbook Table](#d1)  
D.2 [Revenue-at-Risk Analysis](#d2)  

**Part E — Feature Export for NB07*  
E.1 [Merge Cluster Labels to Full User Table](#e1)  
E.2 [Save Enriched Dataset](#e2)  
E.3 [Summary](#e3)  


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from joblib import dump
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

import os
from pathlib import Path
def find_project_root(start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return str(candidate)
    return str(current.parent if current.name == 'notebooks' else current)

BASE = find_project_root()
MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

user_data    = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'user_data.csv'),
                            parse_dates=['signup_date', 'last_active_date', 'first_purchase_date'])
transactions = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'transactions.csv'),
                            parse_dates=['transaction_date'])
transactions = transactions.merge(user_data[['user_id','signup_date']], on='user_id', how='left')
transactions = transactions[transactions['transaction_date'] >= transactions['signup_date']].drop(columns='signup_date')

user_data['churn']           = user_data['churn'].astype(int)
user_data['has_purchased']   = user_data['has_purchased'].astype(int)
user_data['total_sessions']  = user_data['total_sessions'].astype(int)
user_data['total_purchases'] = user_data['total_purchases'].astype(int)

CLUSTER_COLORS = ['#C44E52', '#4C72B0', '#55A868', '#8172B2']
CLUSTER_LABELS = ['VIP / Champions', 'Engaged Regulars', 'Casual Buyers', 'Dormant / At-Risk']

print(f'Users        : {len(user_data):,}')
print(f'Transactions : {len(transactions):,}')
print(f'Buyers       : {user_data["has_purchased"].sum():,} ({user_data["has_purchased"].mean():.1%})')
print(f'Non-buyers   : {(user_data["has_purchased"]==0).sum():,}')
print(f'Txn date range: {transactions["transaction_date"].min().date()} → {transactions["transaction_date"].max().date()}')


Users        : 10,000
Transactions : 15,000
Buyers       : 4,800 (48.0%)
Non-buyers   : 5,200
Txn date range: 2023-01-02 → 2023-06-29


---
## Part A — RFM Segmentation

> **RFM** stands for **Recency** (how recently a user purchased), **Frequency** (how often), and **Monetary** (how much). It is the most widely used framework in customer analytics — simple, interpretable, and directly actionable.

**Why RFM before clustering?**  
RFM gives rule-based, human-readable segments that a marketer or product manager can act on immediately. K-Means (Part B) gives data-driven groupings optimised for compactness — both are useful and complementary.


### A.1 Build RFM Table from Transactions


In [2]:
# Reference date = day after last transaction (standard RFM practice)
transactions_clean = transactions.merge(
    user_data[['user_id', 'signup_date']], on='user_id', how='left'
)
transactions_clean = transactions_clean[
    transactions_clean['transaction_date'] >= transactions_clean['signup_date']
].drop(columns='signup_date')

ref_date = transactions_clean['transaction_date'].max() + pd.Timedelta(days=1)

rfm = transactions_clean.groupby('user_id').agg(
    recency   = ('transaction_date', lambda x: (ref_date - x.max()).days),
    frequency = ('transaction_id', 'count'),
    monetary  = ('amount', 'sum')
).reset_index()

# Merge user attributes for later profiling
rfm = rfm.merge(
    user_data[['user_id', 'engagement_score', 'total_sessions',
               'avg_session_duration', 'churn', 'acquisition_channel',
               'device', 'age', 'gender', 'avg_order_value',
               'days_since_signup', 'churn_eligible']],
    on='user_id', how='left'
)

rfm['eligible_churn'] = np.where(rfm['churn_eligible'].eq(1), rfm['churn'], np.nan)

print(f'RFM table: {len(rfm):,} buying users')
print()
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))


RFM table: 4,800 buying users

       recency  frequency  monetary
count  4800.00    4800.00   4800.00
mean     25.00       3.12   7936.63
std      27.61       1.43   4409.28
min       1.00       1.00    113.00
25%       5.00       2.00   4621.75
50%      15.00       3.00   7445.50
75%      35.00       4.00  10568.50
max     167.00      10.00  28103.00


**Note:** RFM is built from the validated transaction table and reconciled against the user-level purchase flag. Buyer count here should match `user_data['has_purchased'].sum()` exactly (4,800 in this baseline). Any meaningful discrepancy indicates an upstream data contract issue.


### A.2 RFM Score Distribution


In [3]:
# Score each dimension 1-5 using stable quintiles.
# Ties are rank-ordered before qcut so reruns do not fail on duplicate bin edges.
def quantile_score(series, larger_is_better=True):
    if len(series) == 1:
        return pd.Series(3, index=series.index, dtype=int)
    ranked = series.rank(method='first', ascending=larger_is_better)
    q = min(5, len(series))
    raw_score = pd.qcut(ranked, q, labels=False, duplicates='drop') + 1
    if q < 5:
        raw_score = 1 + (raw_score - 1) * (4 / max(q - 1, 1))
    return raw_score.round().clip(1, 5).astype(int)

# Recency: LOWER days = MORE recent = HIGHER score.
# Frequency & Monetary: HIGHER = HIGHER score.
rfm['R'] = quantile_score(rfm['recency'], larger_is_better=False)
rfm['F'] = quantile_score(rfm['frequency'], larger_is_better=True)
rfm['M'] = quantile_score(rfm['monetary'], larger_is_better=True)
rfm['RFM_Total'] = rfm['R'] + rfm['F'] + rfm['M']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

dim_colors = {'R': '#4C72B0', 'F': '#55A868', 'M': '#C44E52'}
for ax, (dim, col) in zip(axes[:3], dim_colors.items()):
    counts = rfm[dim].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=col, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{dim} Score Distribution', fontsize=11, fontweight='bold')
    ax.set_xlabel('Score (1=Worst, 5=Best)')
    ax.set_ylabel('Users')
    for i, v in zip(counts.index, counts.values):
        ax.text(i, v + 10, str(v), ha='center', fontsize=9)

axes[3].hist(rfm['RFM_Total'], bins=13, color='#8172B2',
             edgecolor='white', linewidth=0.5)
axes[3].set_title('RFM Total Score', fontsize=11, fontweight='bold')
axes[3].set_xlabel('Total (3=Worst, 15=Best)')
axes[3].set_ylabel('Users')
axes[3].axvline(rfm['RFM_Total'].mean(), color='black', linestyle='--',
                linewidth=1.5, label=f'Mean: {rfm["RFM_Total"].mean():.1f}')
axes[3].legend(fontsize=9)

plt.suptitle('RFM Score Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### A.3 Rule-Based Segment Assignment


In [4]:
def assign_rfm_segment(row):
    """Assign the first matching priority-ordered RFM segment."""
    r, f, m, total = row['R'], row['F'], row['M'], row['RFM_Total']
    if total >= 13:              return 'Champions'
    elif r >= 4 and f >= 3:      return 'Loyal Customers'
    elif r >= 3 and total >= 9:  return 'Potential Loyalists'
    elif r >= 4 and f <= 2:      return 'Recent Customers'
    elif r <= 2 and f >= 4:      return 'Cannot Lose Them'
    elif r <= 2 and f >= 3:      return 'At Risk'
    elif total <= 6:             return 'Lost'
    else:                        return 'Needs Attention'

rfm['RFM_Segment'] = rfm.apply(assign_rfm_segment, axis=1)

seg_order = ['Champions', 'Loyal Customers', 'Potential Loyalists',
             'Recent Customers', 'Needs Attention', 'Cannot Lose Them',
             'At Risk', 'Lost']
seg_colors = ['#2ca02c', '#1f77b4', '#17becf', '#98df8a',
              '#ff7f0e', '#d62728', '#e377c2', '#7f7f7f']

seg_counts = rfm['RFM_Segment'].value_counts().reindex(seg_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(seg_order[::-1], seg_counts.reindex(seg_order, fill_value=0).values[::-1],
               color=seg_colors[::-1], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of Users')
ax.set_title('RFM Segment Distribution', fontsize=13, fontweight='bold')
for bar, val in zip(bars, seg_counts.reindex(seg_order, fill_value=0).values[::-1]):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({val/len(rfm):.1%})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Segment counts:')
print(rfm['RFM_Segment'].value_counts())


Segment counts:
RFM_Segment
Lost                   1089
Champions               814
Loyal Customers         657
Potential Loyalists     636
Cannot Lose Them        550
At Risk                 384
Recent Customers        372
Needs Attention         298
Name: count, dtype: int64


### A.4 Segment Size & Revenue Profile


In [5]:
seg_profile = rfm.groupby('RFM_Segment').agg(
    users          = ('user_id', 'count'),
    avg_recency    = ('recency', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean'),
    total_revenue  = ('monetary', 'sum'),
    avg_rfm        = ('RFM_Total', 'mean'),
    churn_rate     = ('eligible_churn', 'mean'),
).round(2).reindex(seg_order)
seg_profile['rev_share'] = (seg_profile['total_revenue'] /
                             seg_profile['total_revenue'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

color_map = dict(zip(seg_order, seg_colors))
colors_ordered = [color_map.get(s, '#999') for s in seg_profile.index]

# Avg monetary per segment
axes[0].barh(seg_profile.index[::-1], seg_profile['avg_monetary'][::-1],
             color=colors_ordered[::-1], edgecolor='white', linewidth=0.5)
axes[0].set_title('Avg Spend per User (₹)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('₹')
for i, v in enumerate(seg_profile['avg_monetary'][::-1]):
    axes[0].text(v + 50, i, f'₹{v:,.0f}', va='center', fontsize=8)

# Revenue share
axes[1].barh(seg_profile.index[::-1], seg_profile['rev_share'][::-1],
             color=colors_ordered[::-1], edgecolor='white', linewidth=0.5)
axes[1].set_title('Revenue Share (%)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('%')
for i, v in enumerate(seg_profile['rev_share'][::-1]):
    axes[1].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=8)

# Churn rate
axes[2].barh(seg_profile.index[::-1], seg_profile['churn_rate'][::-1] * 100,
             color=colors_ordered[::-1], edgecolor='white', linewidth=0.5)
overall_churn = rfm.loc[rfm['churn_eligible'].eq(1), 'churn'].mean() * 100
axes[2].axvline(overall_churn, color='black', linestyle='--', linewidth=1,
                label=f'Overall: {overall_churn:.1f}%')
axes[2].set_title('Observable Churn Rate (%)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('%')
axes[2].legend(fontsize=9)
for i, v in enumerate(seg_profile['churn_rate'][::-1] * 100):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8)

plt.suptitle('RFM Segment — Revenue & Churn Profile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(seg_profile[['users','avg_recency','avg_frequency','avg_monetary',
                    'total_revenue','rev_share','churn_rate']].to_string())


                     users  avg_recency  avg_frequency  avg_monetary  total_revenue  rev_share  churn_rate
RFM_Segment                                                                                               
Champions              814         5.45           5.03      13644.51       11106633       29.2        0.17
Loyal Customers        657         4.82           3.32       7494.24        4923715       12.9        0.15
Potential Loyalists    636        11.77           3.25       8783.54        5586332       14.7        0.19
Recent Customers       372         5.03           1.73       3778.09        1405448        3.7        0.14
Needs Attention        298        23.22           2.43       6674.62        1989038        5.2        0.18
Cannot Lose Them       550        40.40           4.34      11056.39        6081013       16.0        0.28
At Risk                384        47.99           3.00       7721.58        2965087        7.8        0.22
Lost                  1089        50.

**Observation:** Champions and Loyal Customers are the smallest segments by user count but generate a disproportionate share of revenue — a textbook Pareto distribution. Lost users are the largest segment, which signals that retention is a more urgent priority than acquisition. The churn rate pattern by segment validates the RFM logic: high-scoring segments should show lower churn, while At-Risk and Lost segments should show the highest churn rates.


### A.5 RFM Heatmap — Recency vs Frequency (coloured by avg Monetary)


In [6]:
# Pivot: R score (rows) vs F score (cols), cell = avg monetary
rfm_pivot = rfm.pivot_table(index='R', columns='F', values='monetary',
                             aggfunc='mean').round(0)
rfm_pivot = rfm_pivot.sort_index(ascending=False)  # R=5 at top

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    rfm_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Avg Monetary (₹)'}, ax=ax
)
ax.set_title('RFM Heatmap — Avg Spend by Recency × Frequency Score\n'
             '(R=5: most recent, F=5: most frequent)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Frequency Score (F)')
ax.set_ylabel('Recency Score (R)')
plt.tight_layout()
plt.show()


**Observation:** The top-right corner (R=5, F=5) represents Champions — users who bought recently and often. This cell should contain the highest average spend. The bottom-left corner (R=1, F=1) are Lost users who haven't transacted in a long time and only ever bought once. The heatmap reveals whether recency or frequency is the stronger driver of monetary value in your dataset.


---
## Part B — K-Means Clustering

> **Why K-Means after RFM?**  
RFM uses fixed rules. K-Means lets the *data* define the boundaries — grouping users by actual distance in feature space, not predetermined thresholds. Both methods have value; K-Means segments are used as features in NB07's churn prediction model.


### B.1 Feature Preparation & Scaling


In [7]:
# Use the same RFM table built in Part A
CLUSTER_FEATURES = ['recency', 'frequency', 'monetary']
X = rfm[CLUSTER_FEATURES].copy()

# Standardise: K-Means is distance-based — unscaled features bias toward high-variance dims
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
dump(scaler, os.path.join(MODELS_DIR, 'rfm_scaler.pkl'))

print('Feature statistics before scaling:')
print(X.describe().round(2))
print()
print('After StandardScaler — mean ≈ 0, std ≈ 1 for each feature:')
print(pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES).describe().round(3))


Feature statistics before scaling:
       recency  frequency  monetary
count  4800.00    4800.00   4800.00
mean     25.00       3.12   7936.63
std      27.61       1.43   4409.28
min       1.00       1.00    113.00
25%       5.00       2.00   4621.75
50%      15.00       3.00   7445.50
75%      35.00       4.00  10568.50
max     167.00      10.00  28103.00

After StandardScaler — mean ≈ 0, std ≈ 1 for each feature:
        recency  frequency  monetary
count  4800.000   4800.000  4800.000
mean      0.000     -0.000     0.000
std       1.000      1.000     1.000
min      -0.869     -1.488    -1.775
25%      -0.724     -0.788    -0.752
50%      -0.362     -0.088    -0.111
75%       0.362      0.613     0.597
max       5.143      4.815     4.574


**Why scale?** Recency is measured in days (range: 0–180), monetary in rupees (range: 100–55,000). Without scaling, monetary dominates the distance calculation purely because of its larger magnitude — the model would cluster primarily on spend and ignore recency. StandardScaler brings all features to the same scale (mean=0, std=1) so each dimension contributes equally.


### B.2 Optimal K — Elbow Method + Silhouette Score


In [8]:
K_range = range(2, 9)
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow curve
axes[0].plot(list(K_range), inertias, 'o-', color='#4C72B0', linewidth=2.5, markersize=8)
axes[0].axvline(4, color='#C44E52', linestyle='--', linewidth=1.5, label='Chosen K=4')
axes[0].set_title('Elbow Method — Inertia vs K', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SSE)')
axes[0].legend(fontsize=10)
for k, v in zip(K_range, inertias):
    axes[0].text(k, v + 100, f'{v:,.0f}', ha='center', fontsize=8)

# Silhouette
bar_colors = ['#C44E52' if k == 4 else '#4C72B0' for k in K_range]
axes[1].bar(list(K_range), sil_scores, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Silhouette Score vs K\n(Higher = Better separated clusters)',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_ylim(0, max(sil_scores) + 0.05)
for k, v in zip(K_range, sil_scores):
    axes[1].text(k, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(facecolor='#C44E52', label='Chosen K=4')], fontsize=10)

plt.suptitle('Part B — Optimal K Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('K | Inertia    | Silhouette')
for k, i, s in zip(K_range, inertias, sil_scores):
    marker = '  ← chosen' if k == 4 else ''
    print(f'{k} | {i:>10.1f} | {s:.4f}{marker}')


K | Inertia    | Silhouette
2 |     8414.2 | 0.3662
3 |     5692.7 | 0.3831
4 |     4365.2 | 0.3381  ← chosen
5 |     3798.9 | 0.3362
6 |     3264.9 | 0.3019
7 |     2874.5 | 0.3039
8 |     2644.1 | 0.2998


**K=4 rationale:** The elbow curve flattens after K=4, and K=4 maps cleanly to four interpretable buyer segments: VIP / Champions, Engaged Regulars, Casual Buyers, and Dormant / At-Risk. Non-buyers are not forced into K-Means; they are assigned a separate rule-based `Non-Buyer` label after clustering so the dashboard shows five total cluster labels.


### B.3 Fit K=4 & Assign Human-Readable Labels


In [9]:
n_clusters = min(4, len(rfm))
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
rfm['cluster'] = km.fit_predict(X_scaled)
dump(km, os.path.join(MODELS_DIR, 'rfm_kmeans.pkl'))

# Profile clusters to assign meaningful names
cluster_stats = rfm.groupby('cluster').agg(
    size         = ('user_id', 'count'),
    avg_recency  = ('recency', 'mean'),
    avg_frequency= ('frequency', 'mean'),
    avg_monetary = ('monetary', 'mean'),
    avg_rfm      = ('RFM_Total', 'mean'),
).round(2).sort_values('avg_rfm', ascending=False)

# Assign labels: sorted by avg_rfm (highest = VIP)
cluster_names = ['VIP / Champions', 'Engaged Regulars', 'Casual Buyers', 'Dormant / At-Risk']
label_map = {cluster_id: cluster_names[i] for i, cluster_id in enumerate(cluster_stats.index)}
rfm['cluster_label'] = rfm['cluster'].map(label_map)
with open(os.path.join(MODELS_DIR, 'rfm_cluster_labels.json'), 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in label_map.items()}, f, indent=2)

print('K-Means cluster profiles (sorted by avg RFM score):')
print(cluster_stats.to_string())
print()
print('Label assignments:')
for k, v in label_map.items():
    row = cluster_stats.loc[k]
    print(f'  Cluster {k} → "{v}" '
          f'(n={row["size"]:,}, recency={row["avg_recency"]:.0f}d, '
          f'freq={row["avg_frequency"]:.1f}, monetary=₹{row["avg_monetary"]:,.0f})')


K-Means cluster profiles (sorted by avg RFM score):
         size  avg_recency  avg_frequency  avg_monetary  avg_rfm
cluster                                                         
3         746        14.18           5.39      15173.84    13.36
0        1824        15.84           3.59       9066.07    10.67
1        1591        17.79           1.95       4333.79     6.62
2         639        81.72           2.09       5234.08     5.07

Label assignments:
  Cluster 3 → "VIP / Champions" (n=746.0, recency=14d, freq=5.4, monetary=₹15,174)
  Cluster 0 → "Engaged Regulars" (n=1,824.0, recency=16d, freq=3.6, monetary=₹9,066)
  Cluster 1 → "Casual Buyers" (n=1,591.0, recency=18d, freq=1.9, monetary=₹4,334)
  Cluster 2 → "Dormant / At-Risk" (n=639.0, recency=82d, freq=2.1, monetary=₹5,234)


### B.4 Cluster Visualisation — PCA 2D Projection


In [10]:
# Reduce 3D RFM space to 2D for visualisation using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
rfm['pca1'] = X_pca[:, 0]
rfm['pca2'] = X_pca[:, 1]

var_explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# PCA scatter by K-Means cluster
for i, label in enumerate(CLUSTER_LABELS):
    mask = rfm['cluster_label'] == label
    axes[0].scatter(rfm.loc[mask, 'pca1'], rfm.loc[mask, 'pca2'],
                   c=CLUSTER_COLORS[i], label=label, alpha=0.4, s=12, edgecolors='none')
axes[0].set_title(f'K-Means Clusters (PCA projection)\n'
                  f'PC1 {var_explained[0]:.1f}% + PC2 {var_explained[1]:.1f}% = '
                  f'{sum(var_explained):.1f}% variance explained',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')
axes[0].legend(markerscale=2, fontsize=9)

# PCA scatter coloured by monetary value
scatter = axes[1].scatter(rfm['pca1'], rfm['pca2'],
                          c=rfm['monetary'], cmap='YlOrRd',
                          alpha=0.4, s=12, edgecolors='none')
plt.colorbar(scatter, ax=axes[1], label='Monetary (₹)')
axes[1].set_title('PCA Projection — Coloured by Monetary Value',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')

plt.suptitle('Part B — K-Means Cluster Visualisation (PCA 2D)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Observation:** PCA compresses the 3-dimensional RFM space into 2 dimensions for visualisation. Clear cluster separation in the PCA plot confirms that K-Means found genuinely distinct groupings — not arbitrary splits of a continuous distribution. The right panel shows that monetary value is the primary driver of separation along PC1, confirming that spend amount is the most differentiating feature in this dataset.


---
## Part C — Cluster Deep-Dive Profiling

> Knowing that clusters exist is not enough. A business needs to understand *who* is in each cluster — their demographics, behaviour, churn risk, and revenue contribution.


### C.1 Revenue & Purchase Behaviour by Cluster


In [11]:
cluster_order = CLUSTER_LABELS
cp = rfm.groupby('cluster_label').agg(
    users         = ('user_id', 'count'),
    avg_recency   = ('recency', 'mean'),
    avg_frequency = ('frequency', 'mean'),
    avg_monetary  = ('monetary', 'mean'),
    total_revenue = ('monetary', 'sum'),
    avg_aov       = ('avg_order_value', 'mean'),
    avg_sessions  = ('total_sessions', 'mean'),
    avg_eng       = ('engagement_score', 'mean'),
    churn_rate    = ('eligible_churn', 'mean'),
).round(2).reindex(cluster_order)
cp['rev_share'] = (cp['total_revenue'] / cp['total_revenue'].sum() * 100).round(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
cluster_colors_ordered = CLUSTER_COLORS

# Avg monetary
axes[0,0].bar(cluster_order, cp['avg_monetary'], color=cluster_colors_ordered,
              edgecolor='white', linewidth=0.5)
axes[0,0].set_title('Avg Total Spend per User (₹)', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('₹')
for i, v in enumerate(cp['avg_monetary']):
    axes[0,0].text(i, v + 100, f'₹{v:,.0f}', ha='center', fontsize=9, fontweight='bold')

# Avg frequency
axes[0,1].bar(cluster_order, cp['avg_frequency'], color=cluster_colors_ordered,
              edgecolor='white', linewidth=0.5)
axes[0,1].set_title('Avg Purchase Frequency', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Transactions')
for i, v in enumerate(cp['avg_frequency']):
    axes[0,1].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

# Revenue share
axes[1,0].pie(cp['rev_share'], labels=cluster_order, autopct='%1.1f%%',
              colors=cluster_colors_ordered, startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1,0].set_title('Revenue Share by Cluster', fontsize=11, fontweight='bold')

# Avg recency
axes[1,1].bar(cluster_order, cp['avg_recency'], color=cluster_colors_ordered,
              edgecolor='white', linewidth=0.5)
axes[1,1].set_title('Avg Recency (Days Since Last Purchase)', fontsize=11, fontweight='bold')
axes[1,1].set_ylabel('Days')
for i, v in enumerate(cp['avg_recency']):
    axes[1,1].text(i, v + 0.5, f'{v:.0f}d', ha='center', fontsize=10, fontweight='bold')

for ax in [axes[0,0], axes[0,1], axes[1,1]]:
    ax.set_xticklabels(cluster_order, rotation=12, ha='right', fontsize=9)

plt.suptitle('Part C — Revenue & Purchase Behaviour by Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(cp[['users','avg_recency','avg_frequency','avg_monetary',
           'total_revenue','rev_share','churn_rate']].to_string())


                   users  avg_recency  avg_frequency  avg_monetary  total_revenue  rev_share  churn_rate
cluster_label                                                                                           
VIP / Champions      746        14.18           5.39      15173.84       11319683       29.7        0.20
Engaged Regulars    1824        15.84           3.59       9066.07       16536515       43.4        0.20
Casual Buyers       1591        17.79           1.95       4333.79        6895057       18.1        0.17
Dormant / At-Risk    639        81.72           2.09       5234.08        3344574        8.8        0.29


**Observation:** VIP/Champions generate the highest average spend and purchase most frequently. Dormant/At-Risk users have the highest recency (longest time since last purchase) — they are slipping away. Despite similar user counts, VIPs and Dormant users represent opposite ends of the revenue spectrum. This asymmetry is the business case for differentiated retention investment.


### C.2 Churn Rate & Engagement by Cluster


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Churn rate
churn_colors = ['#2ca02c' if v < rfm['churn'].mean() else '#d62728'
                for v in cp['churn_rate']]
axes[0].bar(cluster_order, cp['churn_rate'] * 100,
            color=churn_colors, edgecolor='white', linewidth=0.5)
overall_churn = rfm.loc[rfm['churn_eligible'].eq(1), 'churn'].mean() * 100
axes[0].axhline(overall_churn, color='black', linestyle='--', linewidth=1.5,
                label=f'Overall: {overall_churn:.1f}%')
axes[0].set_title('Churn Rate by Cluster (%)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('%')
axes[0].legend(fontsize=9)
for i, v in enumerate(cp['churn_rate'] * 100):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Engagement score
axes[1].bar(cluster_order, cp['avg_eng'], color=CLUSTER_COLORS,
            edgecolor='white', linewidth=0.5)
axes[1].set_title('Avg Engagement Score', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Score (0–1)')
axes[1].set_ylim(0, 1)
for i, v in enumerate(cp['avg_eng']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# Avg sessions
axes[2].bar(cluster_order, cp['avg_sessions'], color=CLUSTER_COLORS,
            edgecolor='white', linewidth=0.5)
axes[2].set_title('Avg Sessions per User', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Sessions')
for i, v in enumerate(cp['avg_sessions']):
    axes[2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

for ax in axes:
    ax.set_xticklabels(cluster_order, rotation=12, ha='right', fontsize=9)

plt.suptitle('Part C — Churn & Engagement by Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Observation:** If churn rates are roughly similar across clusters, it means churn is not strongly predicted by RFM behaviour alone — which is the motivation for NB07's predictive model. Engagement score differences by cluster confirm that behavioural signals (sessions, page views) stratify users in ways that complement pure purchase-history RFM segmentation.


### C.3 Demographic Mix by Cluster


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Device mix
device_pivot = rfm.groupby(['cluster_label', 'device']).size().unstack(fill_value=0)
device_pct   = device_pivot.div(device_pivot.sum(axis=1), axis=0) * 100
device_pct   = device_pct.reindex(cluster_order)
device_pct.plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'],
                edgecolor='white', linewidth=0.5)
axes[0].set_title('Device Mix by Cluster (%)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('%')
axes[0].set_xticklabels(cluster_order, rotation=12, ha='right', fontsize=8)
axes[0].legend(title='Device', fontsize=9)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# Gender mix
gender_pivot = rfm.groupby(['cluster_label', 'gender']).size().unstack(fill_value=0)
gender_pct   = gender_pivot.div(gender_pivot.sum(axis=1), axis=0) * 100
gender_pct   = gender_pct.reindex(cluster_order)
gender_pct.plot(kind='bar', ax=axes[1], color=['#C44E52','#4C72B0'],
                edgecolor='white', linewidth=0.5)
axes[1].set_title('Gender Mix by Cluster (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('%')
axes[1].set_xticklabels(cluster_order, rotation=12, ha='right', fontsize=8)
axes[1].legend(title='Gender', fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Age distribution
for i, label in enumerate(cluster_order):
    ages = rfm[rfm['cluster_label'] == label]['age']
    axes[2].hist(ages, bins=20, alpha=0.55, label=label,
                 color=CLUSTER_COLORS[i], edgecolor='none')
axes[2].set_title('Age Distribution by Cluster', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Users')
axes[2].legend(fontsize=8)

plt.suptitle('Part C — Demographic Mix by Cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Avg age by cluster:')
print(rfm.groupby('cluster_label')['age'].mean().reindex(cluster_order).round(1))


Avg age by cluster:
cluster_label
VIP / Champions      39.5
Engaged Regulars     38.6
Casual Buyers        38.4
Dormant / At-Risk    38.5
Name: age, dtype: float64


### C.4 Cluster Radar Chart — Multi-Metric Comparison


In [14]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normalise cluster metrics 0-1 for radar
radar_metrics = ['avg_recency', 'avg_frequency', 'avg_monetary', 'avg_eng', 'avg_sessions']
radar_labels  = ['Recency\n(lower=better)', 'Frequency', 'Monetary', 'Engagement', 'Sessions']

cp_radar = cp[radar_metrics].copy()
# Invert recency so that 'lower recency' = 'higher score' on chart
cp_radar['avg_recency'] = cp_radar['avg_recency'].max() - cp_radar['avg_recency']
cp_norm = (cp_radar - cp_radar.min()) / (cp_radar.max() - cp_radar.min())

num_vars = len(radar_metrics)
angles   = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles  += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

for i, label in enumerate(cluster_order):
    values = cp_norm.loc[label, radar_metrics].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2,
            color=CLUSTER_COLORS[i], label=label)
    ax.fill(angles, values, alpha=0.1, color=CLUSTER_COLORS[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.set_title('Cluster Radar Chart — Normalised Multi-Metric Comparison',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.show()


**Observation:** The radar chart gives an instant multi-dimensional fingerprint of each cluster. VIP/Champions should dominate on frequency, monetary, and engagement simultaneously. Dormant/At-Risk users should score near-zero on recency (inverted) and frequency while showing higher monetary — they *were* valuable customers who have since gone quiet. This pattern is the most actionable insight in the notebook: dormant high-spenders are the highest-ROI re-engagement target.


---
## Part D — Business Strategy per Segment

> Segmentation without action is just labelling. This section converts cluster insights into a concrete retention playbook — the output a product manager or growth team would actually use.


### D.1 Retention Playbook by Cluster


In [15]:
playbook = {
    'VIP / Champions': {
        'priority'  : '🔴 Protect at all cost',
        'risk'      : 'Low churn, high value — losing one costs ₹20K+',
        'actions'   : [
            'Exclusive loyalty tier with early access to new features',
            'Personalised relationship manager or priority support',
            'High-value referral incentive programme',
            'Annual review / portfolio performance summary email',
        ]
    },
    'Engaged Regulars': {
        'priority'  : '🟠 Upgrade to VIP',
        'risk'      : 'Moderate spend — push them to become Champions',
        'actions'   : [
            'Personalised product recommendations based on purchase history',
            'Milestone rewards at frequency thresholds (e.g., 5th transaction)',
            'Cross-sell complementary investment products',
            'Reactivation nudge if 30+ days since last transaction',
        ]
    },
    'Casual Buyers': {
        'priority'  : '🟡 Activate & educate',
        'risk'      : 'Low frequency — conversion to regular buyer needed',
        'actions'   : [
            'Onboarding / education email series (why invest regularly?)',
            'Low-friction SIP (systematic investment plan) prompts',
            'First-repeat-purchase incentive within 60 days',
            'Engagement content: market updates, portfolio trackers',
        ]
    },
    'Dormant / At-Risk': {
        'priority'  : '⚫ Win-back campaign',
        'risk'      : 'High recency — likely churned, losing ₹5–6K avg',
        'actions'   : [
            'Time-limited win-back offer with zero-fee transaction',
            'Survey: why did you stop? (churn reason collection)',
            'Reactivation push notification at 90-day mark',
            'If no response in 180 days → sunset & reallocate budget',
        ]
    },
}

print('='*72)
print('  RETENTION PLAYBOOK BY CLUSTER')
print('='*72)
for cluster, info in playbook.items():
    row = cp.loc[cluster]
    print(f'\n{cluster}')
    print(f'  Priority    : {info["priority"]}')
    print(f'  Risk note   : {info["risk"]}')
    print(f'  Users       : {int(row["users"]):,}  |  '
          f'Avg spend: ₹{row["avg_monetary"]:,.0f}  |  '
          f'Churn rate: {row["churn_rate"]*100:.1f}%')
    print('  Actions:')
    for action in info['actions']:
        print(f'    → {action}')
print()
print('='*72)


  RETENTION PLAYBOOK BY CLUSTER

VIP / Champions
  Priority    : 🔴 Protect at all cost
  Risk note   : Low churn, high value — losing one costs ₹20K+
  Users       : 746  |  Avg spend: ₹15,174  |  Churn rate: 20.0%
  Actions:
    → Exclusive loyalty tier with early access to new features
    → Personalised relationship manager or priority support
    → High-value referral incentive programme
    → Annual review / portfolio performance summary email

Engaged Regulars
  Priority    : 🟠 Upgrade to VIP
  Risk note   : Moderate spend — push them to become Champions
  Users       : 1,824  |  Avg spend: ₹9,066  |  Churn rate: 20.0%
  Actions:
    → Personalised product recommendations based on purchase history
    → Milestone rewards at frequency thresholds (e.g., 5th transaction)
    → Cross-sell complementary investment products
    → Reactivation nudge if 30+ days since last transaction

Casual Buyers
  Priority    : 🟡 Activate & educate
  Risk note   : Low frequency — conversion to regula

### D.2 Revenue-at-Risk Analysis


In [16]:
# Revenue at risk = dormant users × avg monetary × churn rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue contribution
rev_by_cluster = cp[['total_revenue', 'users']].copy()
rev_by_cluster['revenue_at_risk'] = (
    cp['total_revenue'] * cp['churn_rate']
).round(0)

bars0 = axes[0].bar(cluster_order, rev_by_cluster['total_revenue'] / 1e6,
                    color=CLUSTER_COLORS, edgecolor='white', linewidth=0.5,
                    label='Total Revenue')
bars1 = axes[0].bar(cluster_order, rev_by_cluster['revenue_at_risk'] / 1e6,
                    color='#d62728', alpha=0.5, edgecolor='white', linewidth=0.5,
                    label='Revenue at Risk (churned portion)')
axes[0].set_title('Total Revenue vs Revenue at Risk by Cluster (₹M)',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('₹ Millions')
axes[0].legend(fontsize=9)
axes[0].set_xticklabels(cluster_order, rotation=12, ha='right', fontsize=9)
for bar, val in zip(bars0, rev_by_cluster['total_revenue'] / 1e6):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                f'₹{val:.2f}M', ha='center', fontsize=8, fontweight='bold')

# Budget allocation suggestion
# Cluster priority score: revenue × churn_rate / users (revenue risk per user)
cp['risk_per_user'] = (cp['total_revenue'] * cp['churn_rate'] / cp['users']).round(2)
cp_sorted = cp.sort_values('risk_per_user', ascending=True)
priority_colors = [CLUSTER_COLORS[CLUSTER_LABELS.index(l)] for l in cp_sorted.index]

axes[1].barh(cp_sorted.index, cp_sorted['risk_per_user'],
             color=priority_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Revenue at Risk per User by Cluster\n'
                   '(Higher = higher retention investment priority)',
                   fontsize=11, fontweight='bold')
axes[1].set_xlabel('₹ Revenue at Risk per User')
for i, v in enumerate(cp_sorted['risk_per_user']):
    axes[1].text(v + 10, i, f'₹{v:,.0f}', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Part D — Revenue-at-Risk Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

total_at_risk = rev_by_cluster['revenue_at_risk'].sum()
print(f'Total revenue at risk across all clusters: ₹{total_at_risk:,.0f}')
print(f'As % of total buyer revenue: {total_at_risk / cp["total_revenue"].sum():.1%}')


Total revenue at risk across all clusters: ₹7,713,326
As % of total buyer revenue: 20.2%


**Observation:** Revenue-at-risk quantifies the business case for retention investment. A segment with high churn rate and high average spend has the highest risk-per-user — even a small improvement in retention for that segment returns significant revenue. This chart is the foundation of a retention budget allocation conversation with leadership.


---
## Part E — Feature Export for NB07

> The cluster labels and RFM scores computed in this notebook become **input features** for the churn prediction model in NB07. This is the bridge between segmentation and machine learning.


### E.1 Merge Cluster Labels to Full User Table


In [17]:
# Build the export dataframe: one row per user with a documented RFM schema
rfm_export = rfm[['user_id', 'recency', 'frequency', 'monetary',
                   'R', 'F', 'M', 'RFM_Total', 'RFM_Segment',
                   'cluster', 'cluster_label']].copy()
rfm_export = rfm_export.rename(columns={
    'recency': 'rfm_recency',
    'frequency': 'rfm_frequency',
    'monetary': 'rfm_monetary',
})

# Merge into full user_data (including non-buyers who have no RFM)
user_enriched = user_data.merge(rfm_export, on='user_id', how='left')

# Non-buyers get a dedicated segment
user_enriched['RFM_Segment']   = user_enriched['RFM_Segment'].fillna('Non-Buyer')
user_enriched['cluster_label'] = user_enriched['cluster_label'].fillna('Non-Buyer')
user_enriched['R']             = user_enriched['R'].fillna(0).astype(int)
user_enriched['F']             = user_enriched['F'].fillna(0).astype(int)
user_enriched['M']             = user_enriched['M'].fillna(0).astype(int)
user_enriched['RFM_Total']     = user_enriched['RFM_Total'].fillna(0)
user_enriched['rfm_recency']   = user_enriched['rfm_recency'].fillna(-1)  # -1 = never purchased
user_enriched['rfm_frequency'] = user_enriched['rfm_frequency'].fillna(0)
user_enriched['rfm_monetary']  = user_enriched['rfm_monetary'].fillna(0)

print(f'Enriched user table: {len(user_enriched):,} rows x {len(user_enriched.columns)} columns')
print()
print('New columns added:')
new_cols = ['rfm_recency','rfm_frequency','rfm_monetary','R','F','M','RFM_Total','RFM_Segment','cluster','cluster_label']
for col in new_cols:
    print(f'  {col:20s}: {user_enriched[col].dtype}')
print()
print('Segment distribution in full user table:')
print(user_enriched['cluster_label'].value_counts())


Enriched user table: 10,000 rows x 37 columns

New columns added:
  rfm_recency         : float64
  rfm_frequency       : float64
  rfm_monetary        : float64
  R                   : int64
  F                   : int64
  M                   : int64
  RFM_Total           : float64
  RFM_Segment         : object
  cluster             : float64
  cluster_label       : object

Segment distribution in full user table:
cluster_label
Non-Buyer            5200
Engaged Regulars     1824
Casual Buyers        1591
VIP / Champions       746
Dormant / At-Risk     639
Name: count, dtype: int64


### E.2 Save Enriched Dataset


In [18]:
out_path = os.path.join(BASE, 'data', 'processed', 'user_data_segmented.csv')
user_enriched.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {user_enriched.shape}')
print(f'Columns: {list(user_enriched.columns)}')


Saved: C:\Users\HP\Desktop\Mirae Asset major\data\processed\user_data_segmented.csv
Shape: (10000, 37)
Columns: ['user_id', 'signup_date', 'country', 'state', 'device', 'age', 'gender', 'acquisition_channel', 'is_registered', 'total_sessions', 'avg_session_duration', 'total_pages_viewed', 'last_active_date', 'total_revenue', 'total_purchases', 'avg_order_value', 'first_purchase_date', 'last_purchase_date', 'days_since_signup', 'days_since_last_active', 'avg_revenue_per_purchase', 'revenue_per_session', 'has_purchased', 'engagement_score_raw', 'engagement_score', 'churn_eligible', 'churn', 'rfm_recency', 'rfm_frequency', 'rfm_monetary', 'R', 'F', 'M', 'RFM_Total', 'RFM_Segment', 'cluster', 'cluster_label']


### E.3 Summary


In [19]:
print('='*70)
print('  NOTEBOOK 06 — USER SEGMENTATION SUMMARY')
print('='*70)

print('\n── PART A: RFM SEGMENTS ──')
for seg in rfm['RFM_Segment'].value_counts().index:
    n = (rfm['RFM_Segment'] == seg).sum()
    rev = rfm[rfm['RFM_Segment'] == seg]['monetary'].sum()
    print(f'  {seg:22s}: {n:>5,} users  |  ₹{rev:>12,.0f} revenue')

print('\n── PART B: K-MEANS CLUSTERS (k=4) ──')
for label in CLUSTER_LABELS:
    row = cp.loc[label]
    print(f'  {label:22s}: {int(row["users"]):>5,} users  |  '
          f'avg ₹{row["avg_monetary"]:>7,.0f}  |  '
          f'churn {row["churn_rate"]*100:.1f}%  |  '
          f'{row["rev_share"]:.1f}% of revenue')

print('\n── KEY FINDING ──')
vip_rev_share = cp.loc['VIP / Champions', 'rev_share']
vip_users     = cp.loc['VIP / Champions', 'users']
total_users   = cp['users'].sum()
print(f'  VIP/Champions = {vip_users/total_users:.1%} of buyers but {vip_rev_share:.1f}% of revenue')
print(f'  Total revenue at risk: ₹{(cp["total_revenue"]*cp["churn_rate"]).sum():,.0f}')
print(f'  Enriched dataset saved → user_data_segmented.csv ({len(user_enriched):,} rows)')
print(f'  → Ready for NB07: Predictive Modelling & SHAP Explainability')
print('='*70)


  NOTEBOOK 06 — USER SEGMENTATION SUMMARY

── PART A: RFM SEGMENTS ──
  Lost                  : 1,089 users  |  ₹   4,038,563 revenue
  Champions             :   814 users  |  ₹  11,106,633 revenue
  Loyal Customers       :   657 users  |  ₹   4,923,715 revenue
  Potential Loyalists   :   636 users  |  ₹   5,586,332 revenue
  Cannot Lose Them      :   550 users  |  ₹   6,081,013 revenue
  At Risk               :   384 users  |  ₹   2,965,087 revenue
  Recent Customers      :   372 users  |  ₹   1,405,448 revenue
  Needs Attention       :   298 users  |  ₹   1,989,038 revenue

── PART B: K-MEANS CLUSTERS (k=4) ──
  VIP / Champions       :   746 users  |  avg ₹ 15,174  |  churn 20.0%  |  29.7% of revenue
  Engaged Regulars      : 1,824 users  |  avg ₹  9,066  |  churn 20.0%  |  43.4% of revenue
  Casual Buyers         : 1,591 users  |  avg ₹  4,334  |  churn 17.0%  |  18.1% of revenue
  Dormant / At-Risk     :   639 users  |  avg ₹  5,234  |  churn 29.0%  |  8.8% of revenue

── KEY FINDI

---
## Key Findings

**1. RFM reveals a Lost majority.**  
The largest single RFM segment is Lost: users who scored low on all three dimensions. This is the platform's most urgent retention signal. A win-back campaign targeting this segment even at a 5% recovery rate represents significant revenue.

**2. Champions are a small, critical minority.**  
Champions make up ~20% of buyers but generate a disproportionate share of revenue. Losing a single Champion costs the equivalent of acquiring 10+ Casual Buyers. VIP treatment - exclusive tiers, early access, priority support - is justified by the economics.

**3. Dormant/At-Risk users are former high-spenders.**  
The Dormant cluster has a higher average monetary value than Casual Buyers; they were good customers. This makes them the highest-ROI re-engagement target: less expensive to win back than to acquire new users, and their spend history proves they have intent.

**4. Cluster labels strengthen churn prediction (NB07).**  
The enriched dataset (`user_data_segmented.csv`) includes RFM scores, segment names, and cluster labels as additional features. The churn model in NB07 uses these signals to improve prediction accuracy beyond raw behavioural features alone.
